# aoi-sentinel — Colab Quickstart

Mamba-RL false-call filter for SMT AOI on a free Colab T4 (or Pro L4/A100).

## What you'll run
1. GPU sanity check + clone repo
2. Install deps (torch + mamba-ssm + project) — robust to Colab torch updates
3. Download VisA PCB benchmark data
4. **Stage 0** — supervised pretraining of MambaVision on VisA (~30–60 min on T4)
5. **Stage 1** — Lagrangian PPO on the NPI simulator (~30–60 min)
6. Plot cost / escape / λ trajectories

**Runtime → Change runtime type → GPU** before running.

Tip: free Colab disconnects after ~12h idle. Mount Google Drive and point checkpoint paths there if you want persistence.

## 1. GPU + repo

In [ ]:
!nvidia-smi | head -16

In [ ]:
import os
if not os.path.exists('/content/aoi-sentinel'):
    !git clone https://github.com/DrJinHoChoi/aoi-sentinel.git /content/aoi-sentinel
%cd /content/aoi-sentinel
!git pull

## 2. Install Mamba kernels — robust to Colab torch updates

Colab updates torch every few weeks, which often invalidates pinned wheel versions. We use a 3-tier strategy:

1. **Best**: pip-resolved newest version with `--no-build-isolation` (reuses Colab's torch)
2. **Fallback A**: explicit wheel URL matched to the detected torch + python + CUDA + ABI
3. **Fallback B**: pure-PyTorch build (slower but always works) via `MAMBA_SKIP_CUDA_BUILD=TRUE`

First successful install caches the wheel for subsequent sessions.

In [ ]:
import torch, sys
TORCH_FULL = torch.__version__
TORCH = TORCH_FULL.split('+')[0]
TORCH_MM = '.'.join(TORCH.split('.')[:2])         # e.g. '2.5'
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"  # 'cp311'
CUDA_RAW = (torch.version.cuda or '').replace('.', '')[:3]   # '121'
CUDA_MAJ = (torch.version.cuda or '').split('.')[0] if torch.version.cuda else '12'
ABI_TAG = 'cxx11abiTRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'cxx11abiFALSE'
DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'python : {sys.version_info.major}.{sys.version_info.minor}  ({PY})')
print(f'torch  : {TORCH_FULL}  (mm={TORCH_MM})')
print(f'cuda   : {torch.version.cuda}  (tag=cu{CUDA_MAJ})')
print(f'abi    : {ABI_TAG}')
print(f'device : {DEVICE}')

In [ ]:
# Clean any previous failed attempt and prep build tools.
!pip uninstall -y -q causal-conv1d mamba-ssm 2>/dev/null || true
!pip install -q packaging ninja wheel

In [ ]:
# ============ Tier 1: pip-resolved latest, with --no-build-isolation ============
import subprocess, shlex

def _try(cmd: str) -> bool:
    print(f'>>> {cmd}')
    return subprocess.run(shlex.split(cmd)).returncode == 0

ok = _try('pip install -q --no-build-isolation "causal-conv1d>=1.5.0"') \
     and _try('pip install -q --no-build-isolation "mamba-ssm>=2.2.4"')

if ok:
    print('\n[Tier 1] OK — kernels installed via pip resolver.')
else:
    print('\n[Tier 1] failed — falling back to explicit wheel URLs.')

In [ ]:
# ============ Tier 2: explicit wheel URLs matched to detected env ============
if not ok:
    # NOTE: edit these two version pins if the matching wheel is missing for your env.
    CCONV_VER = '1.5.0.post8'
    MAMBA_VER = '2.2.4'

    # Wheel name shape (as published by upstream releases):
    #   <pkg>-<ver>+cu<MAJ>torch<MM>cxx11abiFALSE-<py>-<py>-linux_x86_64.whl
    base_cconv = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CCONV_VER}'
    base_mamba = f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}'
    cconv_whl = f'{base_cconv}/causal_conv1d-{CCONV_VER}+cu{CUDA_MAJ}torch{TORCH_MM}{ABI_TAG}-{PY}-{PY}-linux_x86_64.whl'
    mamba_whl = f'{base_mamba}/mamba_ssm-{MAMBA_VER}+cu{CUDA_MAJ}torch{TORCH_MM}{ABI_TAG}-{PY}-{PY}-linux_x86_64.whl'

    print(f'try wheel: {cconv_whl}')
    ok = _try(f'pip install -q {cconv_whl}')
    if ok:
        print(f'try wheel: {mamba_whl}')
        ok = _try(f'pip install -q {mamba_whl}')

    if ok:
        print('\n[Tier 2] OK — explicit wheel install succeeded.')
    else:
        print('\n[Tier 2] failed — likely the exact (torch×cuda×py×abi) tuple has no prebuilt wheel.')
        print('Browse releases pages, copy a closer wheel URL, and rerun:')
        print(' - https://github.com/Dao-AILab/causal-conv1d/releases')
        print(' - https://github.com/state-spaces/mamba/releases')

In [ ]:
# ============ Tier 3: pure-PyTorch fallback (slow but always works) ============
if not ok:
    import os
    os.environ['MAMBA_SKIP_CUDA_BUILD'] = 'TRUE'
    os.environ['CAUSAL_CONV1D_SKIP_CUDA_BUILD'] = 'TRUE'
    ok = _try('pip install -q --no-build-isolation causal-conv1d') \
         and _try('pip install -q --no-build-isolation mamba-ssm')
    if ok:
        print('\n[Tier 3] OK — pure-PyTorch path (no custom CUDA kernels).')
        print('Training will be slower but functionally correct.')

assert ok, 'all install tiers failed — check the output above and the release pages.'

In [ ]:
# Project + the rest. Torch is already in Colab; we install the train extras minus torch.
!pip install -q -e "." --no-deps
!pip install -q timm==1.0.11 mambavision albumentations gymnasium tensorboard wandb \
    rich click pyyaml lxml pyarrow pandas opencv-python-headless scikit-learn scikit-image fastapi uvicorn

In [ ]:
# Sanity check: both Mamba surfaces import and run on GPU.
import torch
from mamba_ssm import Mamba
import timm

vm = timm.create_model('mambavision_tiny_1k', pretrained=True).cuda().eval()
x = torch.randn(2, 3, 224, 224).cuda()
with torch.no_grad():
    print('vision out:', vm(x).shape)

mb = Mamba(d_model=128).cuda().eval()
y = torch.randn(2, 64, 128).cuda()
with torch.no_grad():
    print('seq out   :', mb(y).shape)

## 3. Download benchmark data

VisA full archive is ~4.5 GB. We grab it once and prune to the 4 PCB classes (~1.2 GB).

**Optional**: mount Google Drive and copy `data/raw/visa/` there to skip re-downloading on future sessions.

In [ ]:
# Optional Drive mount — uncomment if you want checkpoint/data persistence.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!python scripts/download_visa.py --out data/raw/visa --pcb-only

In [ ]:
!ls data/raw/visa/ && echo '---' && ls data/raw/visa/pcb1/Data/Images/ 2>/dev/null | head

## 4. Stage 0 — pretrain MambaVision on VisA

Cost-sensitive supervised classifier. Output: image-encoder weights for the RL stage.

In [ ]:
!aoi train pretrain --config configs/stage0_pretrain_colab.yaml

## 5. Stage 1 — Mamba RL on the NPI simulator

Lagrangian PPO under the escape-rate budget. Watch for:
- `avg_cost` decreasing over iterations
- `λ` rising when escape budget is violated, falling when satisfied
- `cum_escape` should stay near 0

In [ ]:
!aoi train npi-rl --config configs/stage1_npi_rl_colab.yaml 2>&1 | tee /content/stage1.log

## 6. Plot trajectories

In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
pat = re.compile(r'iter (\d+).*?λ=([\-\d\.]+).*?avg_cost=([\-\d\.]+).*?cum_cost=([\-\d\.]+).*?cum_escape=(\d+)')
with open('/content/stage1.log') as f:
    for line in f:
        m = pat.search(line)
        if m:
            rows.append([int(m[1]), float(m[2]), float(m[3]), float(m[4]), int(m[5])])

import numpy as np
rows = np.array(rows)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col, name in zip(axes, [1, 2, 3, 4], ['λ (Lagrange)', 'avg cost / step', 'cumulative cost', 'cumulative escapes']):
    ax.plot(rows[:, 0], rows[:, col])
    ax.set_xlabel('iter'); ax.set_title(name); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Next

- Run with `--config configs/stage1_npi_rl.yaml` (full size) on a Colab Pro A100 / L4 for the proper baseline numbers.
- Add DeepPCB and SolDef_AI to the data mix (`configs/.../data.dataset`).
- When real Saki data arrives, swap `aoi_sentinel.data.benchmarks` for `aoi_sentinel.data.saki` and rerun stage 1 with `init_from` pointing at the stage 0 checkpoint.